# Prueba técnica Creceré AI — Etapa 1

**Qué hace este notebook:** caracteriza los 100 audios antes de transcribir y deja montado el
benchmark de motores de STT.

| Sección | Pregunta que responde |
|---|---|
| 1. Formato | ¿Qué son estos archivos y qué me obligan a hacer? |
| 2. Pitidos de censura | ¿Dónde está tapado el audio y cuánto? |
| 3. Cruce pitidos ↔ transcripción | ¿Cómo marco esos huecos en el texto? |
| 4. Benchmark Deepgram vs ElevenLabs | ¿Qué motor transcribe mejor *este* audio? |

**Los tres hallazgos que condicionan todo lo demás:**

1. **Los 100 archivos son mono, 8 kHz, PCM 16 bit.** No hay un canal por interlocutor →
   la diarización es obligatoria y su error se propaga a casi toda métrica conversacional.
2. **La censura es un tono sintético de 1000.0 Hz a −15.0 dBFS.** Se detecta con reglas, sin modelos:
   1 623 pitidos tapan el 6.1 % del corpus. No es insumo del STT: es un cruce posterior (sección 3).
3. **El ruido está de un solo lado.** Los audios humanos tienen 12 dB menos de SNR que los de IA
   (9 de 50 bajo 35 dB; 0 de 50 en IA). Si el STT degrada con el ruido, degradará más en el grupo
   humano → la calidad de audio entra al estudio como covariable, no como nota al pie.

> Orden de ejecución: secciones 1–2 corren solas. La 4 necesita `DEEPGRAM_API_KEY` y
> `ELEVENLABS_API_KEY` en el entorno. La 3 se usa después de tener transcripciones.

## 0. Configuración

In [1]:
import json, math, os, time, wave
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path("..")                      # carpeta que contiene las dos carpetas de audios
SAL  = Path("salidas"); SAL.mkdir(exist_ok=True)
CARPETAS = {"humano": "audios_humanos_censurados", "ia": "audios_ia_censurados"}

AUDIOS = [(g, p) for g, c in CARPETAS.items() for p in sorted((ROOT / c).glob("*.wav"))]
print(f"{len(AUDIOS)} audios:", {g: sum(1 for x, _ in AUDIOS if x == g) for g in CARPETAS})

100 audios: {'humano': 50, 'ia': 50}


---
## 1. Formato de los audios

Leer la cabecera WAV basta: `wave` de la librería estándar da sample rate, canales, profundidad
de bits y número de muestras sin cargar el audio a memoria.

- hz: para ver el formato.
- canales: de pronto agente esta a un lado cliente al otro.
- bits
- seg: duracion del audio en segundos

In [2]:
def cabecera(p):
    with wave.open(str(p)) as w:
        return {"hz": w.getframerate(), "canales": w.getnchannels(),
                "bits": 8 * w.getsampwidth(), "seg": round(w.getnframes() / w.getframerate(), 1)}

fmt = pd.DataFrame([{"grupo": g, "file": p.name, **cabecera(p)} for g, p in AUDIOS])
fmt.groupby(["hz", "canales", "bits"]).size().rename("archivos").to_frame() # agrupo los que tienen las mismas caracteristicas tecnicas

,,,archivos
hz,canales,bits,
8000,1,16,100


**8 000 Hz · 1 canal · 16 bit en los 100 archivos.** Basicamente me dice que todos tienen las mismas caracteristicas técnicas



- **Mono ⇒ hay que diarizar.** No existe la opción barata de separar agente y cliente por pista.

In [3]:
fmt.groupby("grupo")["seg"].describe()[["count", "mean", "50%", "min", "max"]].round(1)

,count,mean,50%,min,max
grupo,,,,,
humano,50.0,238.1,171.1,74.6,1233.2
ia,50.0,219.3,149.0,60.3,561.6


**En min:**

In [4]:
resumen = fmt.groupby("grupo")["seg"].describe()[["count", "mean", "50%", "min", "max"]]

resumen[["mean", "50%", "min", "max"]] = (
    resumen[["mean", "50%", "min", "max"]] / 60
).round(1)

print(resumen)

        count  mean  50%  min   max
grupo                              
humano   50.0   4.0  2.9  1.2  20.6
ia       50.0   3.7  2.5  1.0   9.4


#### Mann–Whitney

Aunque la mediana de duración fue menor en las llamadas de IA (149.0 s vs. 171.1 s), la diferencia no fue estadísticamente significativa según la prueba U de Mann–Whitney (p = 0.32).

In [5]:
from scipy.stats import mannwhitneyu

humano = fmt[fmt["grupo"] == "humano"]["seg"]
ia = fmt[fmt["grupo"] == "ia"]["seg"]

u, p = mannwhitneyu(humano, ia, alternative="two-sided")

print("U =", u)
print("p =", p)

if p < 0.05:
    print("Diferencia estadísticamente significativa")
else:
    print("No hay diferencia estadísticamente significativa")

U = 1395.0
p = 0.3191709781105352
No hay diferencia estadísticamente significativa


---
## 2. Pitidos de censura

El proceso tiene tres fases separadas, para que la validación no sea un espejo de la calibración:

1. **Calibración** — escucho una muestra de audios, anoto a mano dónde están los pitidos y mido
   sus propiedades. De ahí salen los umbrales del detector.
2. **Detección** — aplico esos umbrales a los 100 audios completos, sin mirar manualmente dónde
   están los pitidos.
3. **Validación** — tomo una muestra de las detecciones automáticas (de audios que **no** usé
   para calibrar) y la compruebo escuchándola.

### Fase 1 — Calibración

No elegí las condiciones antes de escuchar nada. Escuché 4 audios completos (`7a59be10`,
`32616618` del grupo IA; `380f5075`, `336a8602` del grupo humano) y anoté a mano el segundo
exacto de cada pitido que oía — 24 marcas en total. Después medí, en una ventana alrededor de
cada marca, qué frecuencia dominaba, qué tan concentrada estaba la energía ahí, qué tan plana
era la amplitud y cuánto duraba

In [6]:
def _leer(path):
    with wave.open(str(path)) as w:
        x = np.frombuffer(w.readframes(w.getnframes()), np.int16).astype(np.float32) / 32768
        return x, w.getframerate()

def _frame_stats(x, sr):
    N, H = int(.025 * sr), int(.010 * sr)
    fr  = np.lib.stride_tricks.sliding_window_view(x, N)[::H]
    esp = np.abs(np.fft.rfft(fr * np.hanning(N))) ** 2
    hz  = np.fft.rfftfreq(N, 1 / sr)
    dom = esp.argmax(1)
    conc = np.array([esp[i, max(0, d - 1):d + 2].sum() / (esp[i].sum() + 1e-12) for i, d in enumerate(dom)])
    rms  = np.sqrt((fr ** 2).mean(1))
    return hz[dom], conc, rms, H

def medir_marca(carpeta, archivo, a, b, margen=3.5):
    """Busca, alrededor del segundo anotado a mano, el tramo MÁS LARGO con energía concentrada
    en una sola frecuencia y que no sea silencio"""
    x, sr = _leer(ROOT / carpeta / archivo)
    hz, conc, rms, H = _frame_stats(x, sr)
    i0, i1 = int(max(0, a - margen) * sr / H), min(int((b + margen) * sr / H), len(conc))
    hz, conc, rms = hz[i0:i1], conc[i0:i1], rms[i0:i1]

    # conc > .9 y rms > .03 son filtros preliminares y exploratorios, solo para esta fase de
    # calibración: sirven para descartar silencio y ubicar el segmento tonal más energético
    # alrededor de cada marca manual. NO son los umbrales finales del detector -- esos
    # (conc_min=.95, cv_max=.20, dur_min=.10) se definen más abajo, en pitidos(), a partir de
    # lo que se mide aquí.
    alto, runs, i = (conc > .9) & (rms > .03), [], 0
    while i < len(alto):
        if not alto[i]:
            i += 1; continue
        j = i
        while j + 1 < len(alto) and alto[j + 1]:
            j += 1
        runs.append((i, j)); i = j + 1
    j0, j1 = max(runs, key=lambda r: r[1] - r[0])        # el tramo más largo, no el más fuerte
    seg_rms = rms[j0:j1 + 1]
    return {"archivo": archivo[:8], "marca": f"{a}-{b}s", "freq_hz": round(float(np.median(hz[j0:j1 + 1]))),
            "concentracion": round(float(conc[j0:j1 + 1].min()), 3), "dur_ms": round((j1 - j0 + 1) * H / sr * 1000),
            "cv_amplitud": round(float(seg_rms.std() / seg_rms.mean()), 3),
            "dbfs": round(float(20 * np.log10(seg_rms.mean() + 1e-12)), 1)}

MARCAS_MANUALES = {
    ("audios_ia_censurados", "7a59be10-fc3c-423b-8619-a13aac8ffea3.wav"):
        [(2, 3), (13, 14), (17, 18), (25, 26), (34, 35), (55, 56), (59, 60)],
    ("audios_ia_censurados", "32616618-6ce3-484f-a1d9-c014394a3492.wav"):
        [(0, 1), (6, 7), (12, 13), (26, 26), (38, 38), (63, 64)],
    ("audios_humanos_censurados", "380f5075-47be-435f-b595-7e2c48dbb21b.wav"):
        [(11, 12), (14, 14), (20, 20), (72, 72)],
    ("audios_humanos_censurados", "336a8602-1b56-4ca6-9dcc-7491b9a92a8a.wav"):
        [(7, 7), (8, 9), (15, 15), (47, 47), (52, 52), (65, 65), (68, 68)],
}

medidas = pd.DataFrame([medir_marca(carpeta, archivo, a, b)
                        for (carpeta, archivo), pts in MARCAS_MANUALES.items() for a, b in pts])
display(medidas)
medidas[["freq_hz", "concentracion", "cv_amplitud", "dur_ms", "dbfs"]].describe().round(3)

,archivo,marca,freq_hz,concentracion,dur_ms,cv_amplitud,dbfs
0,7a59be10,2-3s,1000,0.979,770,0.022,-15.0
1,7a59be10,13-14s,1000,0.979,1090,0.019,-15.0
2,7a59be10,17-18s,1000,0.979,1670,0.015,-15.0
3,7a59be10,25-26s,1000,0.979,890,0.021,-15.0
4,7a59be10,34-35s,1000,0.979,1190,0.018,-15.0
5,7a59be10,55-56s,1000,0.979,1330,0.017,-15.0
6,7a59be10,59-60s,1000,0.979,910,0.020,-15.0
7,32616618,0-1s,1000,0.978,850,0.020,-15.0
8,32616618,6-7s,1000,1.000,1040,0.007,-15.0
9,32616618,12-13s,1000,0.979,1690,0.015,-15.0


,freq_hz,concentracion,cv_amplitud,dur_ms,dbfs
count,24.0,24.000,24.000,24.000,24.000
mean,1000.0,0.973,0.021,995.417,-15.021
std,0.0,0.020,0.010,388.352,0.041
min,1000.0,0.911,0.007,470.000,-15.100
25%,1000.0,0.979,0.016,725.000,-15.000
50%,1000.0,0.979,0.020,900.000,-15.000
75%,1000.0,0.979,0.022,1350.000,-15.000
max,1000.0,1.000,0.048,1690.000,-15.000


**Lo que salió, sobre las 24 marcas anotadas de oído:**

| Medida | Rango observado | Umbral que fijé (con margen) |
|---|---|---|
| Frecuencia dominante | **1000 Hz exactos en las 24** | ±40 Hz alrededor de 1000 |
| Concentración de energía en esa frecuencia | 0.911 – 1.000 (23/24 ≥ 0.95) | > 95 % |
| Planitud de amplitud (CV de la envolvente RMS) | 0.007 – 0.048 | CV ≤ 0.20 (≈4× el máximo que vi) |
| Duración continua | 0.47 s – 1.69 s | ≥ 100 ms (bajo el mínimo real, para no perder pitidos cortos) |

Y por qué la voz no dispara esto: una vocal reparte su energía entre varios armónicos (F0 entre
70 y 320 Hz) y modula la amplitud todo el tiempo, así que no se sostiene ni concentrada ni plana
durante 100 ms seguidos — es la contraparte que necesitaba para saber que los umbrales de arriba
no son arbitrarios, separan lo que oí de la voz alrededor.

Con esos cuatro umbrales queda definido `pitidos()`, el detector que se usará en la Fase 2.

In [7]:
def leer(p):
    with wave.open(str(p)) as w:
        x = np.frombuffer(w.readframes(w.getnframes()), np.int16).astype(np.float32) / 32768
        return x, w.getframerate()

def pitidos(x, sr, f_obj=1000, conc_min=.95, cv_max=.20, dur_min=.10):
    """Devuelve [(inicio_s, fin_s, hz, dbfs)] de cada pitido de censura."""
    N, H = int(.025 * sr), int(.010 * sr)                        # ventana 25 ms, salto 10 ms
    fr  = np.lib.stride_tricks.sliding_window_view(x, N)[::H]
    esp = np.abs(np.fft.rfft(fr * np.hanning(N))) ** 2
    hz  = np.fft.rfftfreq(N, 1 / sr)
    dom = esp.argmax(1)

    vecinos = np.clip(dom[:, None] + [-1, 0, 1], 0, esp.shape[1] - 1)   # el bin dominante ±1
    conc    = np.take_along_axis(esp, vecinos, 1).sum(1) / (esp.sum(1) + 1e-12)
    es_tono = (conc > conc_min) & (abs(hz[dom] - f_obj) <= 40)          # condiciones 1 y 2
    rms     = np.sqrt((fr ** 2).mean(1))

    out, i = [], 0
    while i < len(es_tono):
        if not es_tono[i]:
            i += 1; continue
        j = i                                                   # extiende la racha de frames tonales
        while j + 1 < len(es_tono) and es_tono[j + 1]:
            j += 1
        amp = rms[i:j + 1]
        if (j - i + 1) * H / sr >= dur_min and amp.std() / amp.mean() <= cv_max:   # condiciones 3 y 4
            out.append((round(i * H / sr, 2), round((j * H + N) / sr, 2),
                        round(float(np.median(hz[dom[i:j + 1]])), 1),
                        round(float(20 * np.log10(amp.mean())), 1)))
        i = j + 1
    return out

### Fase 2 — Detección

Con los umbrales ya fijados en la Fase 1, se corre `pitidos()` sobre los 100 audios completos
(incluidos los 4 que se usaron para calibrar) sin volver a mirar manualmente dónde están los
pitidos.

In [8]:
censura = {p.name: pitidos(*leer(p)) for _, p in AUDIOS}
json.dump(censura, open(SAL / "censura.json", "w"))

ev = pd.DataFrame([e for v in censura.values() for e in v], columns=["ini", "fin", "hz", "dbfs"])
ev["dur"] = ev.fin - ev.ini
print(f"{len(ev)} pitidos | frecuencias distintas: {ev.hz.unique()} Hz | "
      f"nivel: {ev.dbfs.min()} a {ev.dbfs.max()} dBFS")
print(f"duración del pitido -> mediana {ev.dur.median():.2f}s  p90 {ev.dur.quantile(.9):.2f}s  max {ev.dur.max():.2f}s")

1623 pitidos | frecuencias distintas: [1000.] Hz | nivel: -15.1 a -15.0 dBFS
duración del pitido -> mediana 0.78s  p90 1.30s  max 6.26s


**Una sola frecuencia y un solo nivel en los 1 623 eventos.** Esa es una validación *interna* del
detector (si estuviera confundiendo voz con pitido, vería dispersión en ambas columnas), pero no
reemplaza la Fase 3: escuchar una muestra independiente.

### Fase 3 — Validación

Validar con los mismos 4 audios de la Fase 1 no dice nada nuevo: ya sabíamos que ahí el detector
funciona, porque de ahí salieron los umbrales. La muestra de validación tiene que salir de los
**otros 96 audios**, elegida al azar sobre lo que el detector encontró solo, sin que yo mirara antes.

Cada evento de la muestra se recorta a un `.wav` corto (evento + 1 s de margen a cada lado) en
`salidas/validacion_clips/` para poder escucharlo suelto, y la muestra completa queda en
`salidas/validacion_muestra.csv` con una columna `correcto` vacía para llenar a mano.

In [9]:
def exportar_clip(carpeta, archivo, ini, fin, pad=1.0, out_dir=SAL / "validacion_clips"):
    """Recorta el evento +/- pad segundos a un wav aparte, para poder escucharlo suelto."""
    out_dir.mkdir(parents=True, exist_ok=True)
    x, sr = _leer(ROOT / carpeta / archivo)
    i0, i1 = max(0, int((ini - pad) * sr)), min(len(x), int((fin + pad) * sr))
    seg = (x[i0:i1] * 32768).clip(-32768, 32767).astype(np.int16)
    nombre = f"{archivo[:8]}_{ini:.1f}-{fin:.1f}.wav"
    with wave.open(str(out_dir / nombre), "w") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(seg.tobytes())
    return nombre

AUDIOS_CALIBRACION = {archivo for _, archivo in MARCAS_MANUALES}   # ya se usaron para fijar los umbrales: fuera de la validación

detecciones = pd.DataFrame([{"grupo": g, "file": p.name, "ini": ini, "fin": fin, "hz": hz, "dbfs": dbfs}
                            for g, p in AUDIOS if p.name not in AUDIOS_CALIBRACION
                            for ini, fin, hz, dbfs in censura[p.name]])

RUTA_VALIDACION = SAL / "validacion_muestra.csv"
if RUTA_VALIDACION.exists():
    muestra = pd.read_csv(RUTA_VALIDACION)          # no se vuelve a sortear si ya existe (para no perder lo marcado a mano)
else:
    # se itera el groupby en vez de usar .apply() -- en pandas recientes .apply() puede excluir
    # la columna de agrupación del resultado, y aquí sí la necesitamos
    muestra = pd.concat([d.sample(n=10, random_state=0) for _, d in detecciones.groupby("grupo")]).reset_index(drop=True)
    muestra["dur"] = (muestra.fin - muestra.ini).round(2)
    muestra["clip"] = [exportar_clip(CARPETAS[g], f, i, j)
                       for g, f, i, j in zip(muestra.grupo, muestra.file, muestra.ini, muestra.fin)]
    muestra["correcto"] = np.nan                    # completar después de escuchar: 1 = sí es censura, 0 = falso positivo
    muestra.to_csv(RUTA_VALIDACION, index=False)

print(f"{len(muestra)} clips en {SAL / 'validacion_clips'} -- escúchalos y completa 'correcto' en {RUTA_VALIDACION}")
muestra

20 clips en salidas\validacion_clips -- escúchalos y completa 'correcto' en salidas\validacion_muestra.csv


,grupo,file,ini,fin,hz,dbfs,dur,clip,correcto
0,humano,b954108b-c0c2-45e8-9032-1a3e87d60778.wav,71.87,72.69,1000.0,-15.0,0.82,b954108b_71.9-72.7.wav,NaN
1,humano,74b16f62-f04d-42c8-9872-ea9b9ed1dd75.wav,586.72,587.26,1000.0,-15.0,0.54,74b16f62_586.7-587.3.wav,NaN
2,humano,92778da9-c37a-4eee-9ca5-81f835e7c5a4.wav,70.76,71.44,1000.0,-15.0,0.68,92778da9_70.8-71.4.wav,NaN
3,humano,e98181dc-2d7b-41ea-9524-59c721651a39.wav,20.15,20.89,1000.0,-15.0,0.74,e98181dc_20.1-20.9.wav,NaN
4,humano,1e3beb83-824b-4ad5-8483-f4ff6ea3c05e.wav,76.44,78.22,1000.0,-15.0,1.78,1e3beb83_76.4-78.2.wav,NaN
5,humano,ae0573ba-193e-42ac-bcc8-4c376a122f8e.wav,34.39,34.94,1000.0,-15.0,0.55,ae0573ba_34.4-34.9.wav,NaN
6,humano,e76a6feb-b941-4dfc-aebf-4288c13c539c.wav,34.12,35.02,1000.0,-15.0,0.90,e76a6feb_34.1-35.0.wav,NaN
7,humano,50e0f632-d195-4476-b049-3720482d65d9.wav,52.57,53.40,1000.0,-15.0,0.83,50e0f632_52.6-53.4.wav,NaN
8,humano,ae0573ba-193e-42ac-bcc8-4c376a122f8e.wav,236.52,238.34,1000.0,-15.0,1.82,ae0573ba_236.5-238.3.wav,NaN
9,humano,0d2d37dd-38a4-40dd-ac62-b430f722ed57.wav,345.53,346.24,1000.0,-15.0,0.71,0d2d37dd_345.5-346.2.wav,NaN


**Ahora escucha los 20 clips** (10 por grupo) en `salidas/validacion_clips/` y en
`salidas/validacion_muestra.csv` completa la columna `correcto`: `1` si de verdad es un pitido de
censura, `0` si es un falso positivo (el detector se equivocó). Guarda el CSV y corre la celda
de abajo.

In [ ]:
val = pd.read_csv(RUTA_VALIDACION)
if val["correcto"].isna().any():
    print(f"Faltan {val['correcto'].isna().sum()} de {len(val)} filas por marcar en {RUTA_VALIDACION}.")
else:
    precision = val["correcto"].mean()
    print(f"Precisión sobre la muestra validada a mano: {precision*100:.1f}% ({int(val.correcto.sum())}/{len(val)})")
    if precision < 1:
        print("Falsos positivos:")
        display(val[val.correcto == 0])

In [6]:
cen = pd.DataFrame([{"grupo": g, "file": p.name,
                     "pitidos": len(censura[p.name]),
                     "seg_tapados": round(sum(b - a for a, b, *_ in censura[p.name]), 1)}
                    for g, p in AUDIOS]).merge(fmt[["file", "seg"]], on="file")
cen["pct_tapado"] = (cen.seg_tapados / cen.seg * 100).round(2)

resumen = cen.groupby("grupo").agg(audios=("file", "count"), pitidos=("pitidos", "sum"),
                                   min_tapados=("seg_tapados", lambda s: round(s.sum() / 60, 1)),
                                   pct_mediano=("pct_tapado", "median"))
display(resumen)
print("Peor caso:", cen.loc[cen.pct_tapado.idxmax(), ["file", "grupo", "pct_tapado"]].to_dict())

,audios,pitidos,min_tapados,pct_mediano
grupo,,,,
humano,50,647,9.0,4.995
ia,50,976,14.3,8.385


Peor caso: {'file': '631a4d16-0c07-4215-9476-ff9c8179413a.wav', 'grupo': 'ia', 'pct_tapado': 21.01}


La IA tiene casi el doble de censura que los humanos (8.4 % vs 5.0 % del audio, p < 1e-5).
La lectura más plausible no es que hable de cosas más sensibles, sino que **verbaliza más datos
del titular**: saludo con nombre completo, confirmación de documento, lectura de saldo.
Es una hipótesis que la transcripción podrá confirmar mirando qué hay alrededor de cada pitido.

---
## 3. Cruce pitidos ↔ transcripción

Aquí es donde sirve la sección 2. El STT no sabe que hubo un pitido: o lo ignora, o **inventa
una palabra encima**. Con los tiempos ya calculados se hacen dos cosas de una vez:

1. insertar `[CENSURADO]` en el lugar exacto del hueco, y
2. **descartar** cualquier palabra cuyo intervalo se solape con un pitido — esas son alucinaciones.

Se ejecuta sobre la salida normalizada de la sección 4 (lista de `{text, start, end, speaker}`).

In [10]:
def marcar_censura(words, beeps, etiqueta="[CENSURADO]"):
    """Intercala marcadores de censura y elimina lo que el motor alucinó sobre el pitido."""
    ev, out, k = [(a, b) for a, b, *_ in beeps], [], 0
    for w in words:
        while k < len(ev) and ev[k][1] <= w["start"]:            # pitidos que ya quedaron atrás
            out.append({**w, "text": etiqueta, "start": ev[k][0], "end": ev[k][1]}); k += 1
        if not any(a < w["end"] and w["start"] < b for a, b in ev):
            out.append(w)
    return out + [{"text": etiqueta, "start": a, "end": b, "speaker": ""} for a, b in ev[k:]]


def por_turnos(words, max_pausa=1.0):
    """Agrupa palabras en turnos de hablante -> texto legible con marca de tiempo."""
    turnos = []
    for w in words:
        if turnos and w["speaker"] == turnos[-1]["speaker"] and w["start"] - turnos[-1]["fin"] < max_pausa:
            turnos[-1]["texto"] += " " + w["text"]; turnos[-1]["fin"] = w["end"]
        else:
            turnos.append({"ini": w["start"], "fin": w["end"], "speaker": w["speaker"], "texto": w["text"]})
    return "\n".join(f"[{int(t['ini'])//60:02d}:{int(t['ini'])%60:02d}] {t['speaker']}: {t['texto']}"
                     for t in turnos)

In [11]:
# ejemplo mínimo: el motor transcribió "Bancolombia" encima de un pitido que va de 1.2 a 2.0 s
demo_words = [{"text": "Buenas", "start": 0.2, "end": 0.6, "speaker": "S0"},
              {"text": "tardes", "start": 0.6, "end": 1.1, "speaker": "S0"},
              {"text": "Bancolombia", "start": 1.3, "end": 1.9, "speaker": "S0"},
              {"text": "con", "start": 2.1, "end": 2.3, "speaker": "S0"},
              {"text": "usted", "start": 2.3, "end": 2.7, "speaker": "S0"}]
print(por_turnos(marcar_censura(demo_words, [(1.2, 2.0, 1000.0, -15.0)])))

[00:00] S0: Buenas tardes [CENSURADO] con usted


---
## 4. Benchmark: Deepgram vs ElevenLabs

**Diseño.** 6 audios (3 del grupo IA + 3 del grupo humano) × 2 motores = 12 transcripciones. De
cada grupo se toma el audio más corto, el de duración mediana y el más largo — así el benchmark
compara los dos motores **dentro de cada grupo** y no queda sesgado hacia el grupo que domine el
extremo (el humano tenía el audio más largo de todo el corpus; con 3 audios globales, la IA se
quedaba sin su propio caso de estrés).

**Coste:** ~38 min de audio × 2 corridas ≈ 1.3 h facturadas ≈ **US$0.36 en total.**

Mismo modelo, mismo idioma, misma diarización, mismos 2 hablantes en ambos motores — lo único que
cambia es cuál de los dos transcribe.

In [12]:
BENCH = []
for g in ["ia", "humano"]:
    sel = fmt[fmt.grupo == g].sort_values("seg")
    BENCH += [(f"{g}_corto", sel.iloc[0]), (f"{g}_mediano", sel.iloc[len(sel) // 2]), (f"{g}_largo", sel.iloc[-1])]

rutas = {k: next(p for _, p in AUDIOS if p.name == r.file) for k, r in BENCH}

pd.DataFrame([{"caso": k, "file": r.file[:8], "grupo": r.grupo, "seg": r.seg,
               "pitidos": len(censura[r.file])} for k, r in BENCH])

,caso,file,grupo,seg,pitidos
0,ia_corto,7a59be10,ia,60.3,8
1,ia_mediano,3e8b2986,ia,156.1,11
2,ia_largo,79c5d6ae,ia,561.6,41
3,humano_corto,380f5075,humano,74.6,4
4,humano_mediano,93d0abfc,humano,175.5,3
5,humano_largo,ce33bc23,humano,1233.2,28


### 4.2 Llamadas a las dos APIs

Cada función devuelve el JSON crudo; `norm_*` lo traduce al mismo formato
(`{text, start, end, speaker, conf}`) para poder comparar peras con peras.

In [13]:
import requests

DG_KEY = os.getenv("DEEPGRAM_API_KEY", "")
EL_KEY = os.getenv("ELEVENLABS_API_KEY", "")

# Si el entorno no está tomando la key nueva (el kernel se inició antes de que la exportaras),
# pégala aquí directo y corre esta celda de nuevo -- así no depende de reiniciar nada:
# EL_KEY = "sk_..."

print(f"DEEPGRAM_API_KEY: {'ok' if DG_KEY else 'FALTA'} (len={len(DG_KEY)})")
print(f"ELEVENLABS_API_KEY: {'ok' if EL_KEY else 'FALTA'} "
      f"(len={len(EL_KEY)}, empieza en 'sk_': {EL_KEY.startswith('sk_')})")


def deepgram(path):
    q = ("model=nova-3&language=multi&diarize=true&punctuate=true"
         "&smart_format=true&utterances=true")
    r = requests.post(f"https://api.deepgram.com/v1/listen?{q}",
                      headers={"Authorization": f"Token {DG_KEY}", "Content-Type": "audio/wav"},
                      data=path.read_bytes(), timeout=1200)
    if not r.ok:
        raise RuntimeError(f"{r.status_code} {r.reason}: {r.text[:500]}")
    return r.json()


def elevenlabs(path):
    data = [("model_id", "scribe_v2"), ("language_code", "spa"), ("diarize", "true"),
            ("num_speakers", "2"), ("timestamps_granularity", "word")]
    audio = {"file": (path.name, path.read_bytes(), "audio/wav")}
    r = requests.post("https://api.elevenlabs.io/v1/speech-to-text",
                      headers={"xi-api-key": EL_KEY}, data=data, files=audio, timeout=1800)
    if not r.ok:
        raise RuntimeError(f"{r.status_code} {r.reason}: {r.text[:500]}")
    return r.json()


def norm_deepgram(j):
    ws = j["results"]["channels"][0]["alternatives"][0]["words"]
    return [{"text": w.get("punctuated_word", w["word"]), "start": w["start"], "end": w["end"],
             "speaker": f"S{w.get('speaker', 0)}", "conf": w.get("confidence")} for w in ws]


def norm_elevenlabs(j):
    return [{"text": w["text"], "start": w["start"], "end": w["end"],
             "speaker": w.get("speaker_id", "S0"),
             "conf": math.exp(w["logprob"]) if w.get("logprob") is not None else None}
            for w in j["words"] if w.get("type") == "word"]


MOTORES = {"deepgram": (deepgram, norm_deepgram), "elevenlabs": (elevenlabs, norm_elevenlabs)}

DEEPGRAM_API_KEY: ok (len=40)
ELEVENLABS_API_KEY: ok (len=51, empieza en 'sk_': True)


### 4.3 Correr las 12 transcripciones

In [15]:
def correr():
    res = {}
    for caso, path in rutas.items():
        for motor, (llamar, norm) in MOTORES.items():
            nombre = f"{caso}__{motor}"
            crudo = SAL / f"{nombre}.json"
            if crudo.exists():                       # no se vuelve a pagar lo ya transcrito
                res[nombre] = json.loads(crudo.read_text()); continue
            t0 = time.time()
            try:
                j = llamar(path)
            except Exception as e:
                print(f"  ✗ {nombre}: {e}"); continue
            words = norm(j)
            res[nombre] = words
            crudo.write_text(json.dumps(words, ensure_ascii=False))
            (SAL / f"{nombre}.txt").write_text(
                por_turnos(marcar_censura(words, censura[path.name])), encoding="utf-8")
            print(f"  ✓ {nombre}: {len(words)} palabras en {time.time()-t0:.0f}s")
    return res

if DG_KEY and EL_KEY:
    RES = correr()
else:
    RES = {}
    print("Falta definir DEEPGRAM_API_KEY y/o ELEVENLABS_API_KEY en el entorno.\n"
          "  Windows PowerShell:  $env:DEEPGRAM_API_KEY='...'   antes de abrir el notebook\n"
          "  o desde aquí:        os.environ['DEEPGRAM_API_KEY']='...'")

  ✗ ia_corto__elevenlabs: 400 Client Error: Bad Request for url: https://api.elevenlabs.io/v1/speech-to-text
  ✓ ia_mediano__deepgram: 237 palabras en 2s
  ✗ ia_mediano__elevenlabs: 400 Client Error: Bad Request for url: https://api.elevenlabs.io/v1/speech-to-text
  ✓ ia_largo__deepgram: 1369 palabras en 3s
  ✗ ia_largo__elevenlabs: 400 Client Error: Bad Request for url: https://api.elevenlabs.io/v1/speech-to-text
  ✓ humano_corto__deepgram: 181 palabras en 2s
  ✗ humano_corto__elevenlabs: 400 Client Error: Bad Request for url: https://api.elevenlabs.io/v1/speech-to-text
  ✓ humano_mediano__deepgram: 310 palabras en 2s
  ✗ humano_mediano__elevenlabs: 400 Client Error: Bad Request for url: https://api.elevenlabs.io/v1/speech-to-text
  ✓ humano_largo__deepgram: 2869 palabras en 6s
  ✗ humano_largo__elevenlabs: 400 Client Error: Bad Request for url: https://api.elevenlabs.io/v1/speech-to-text


In [15]:
def correr_elevenlabs():
    """Reintenta solo ElevenLabs -- deepgram ya quedó cacheado en salidas/*.json."""
    res = {}
    for caso, path in rutas.items():
        nombre = f"{caso}__elevenlabs"
        crudo = SAL / f"{nombre}.json"
        if crudo.exists():
            res[nombre] = json.loads(crudo.read_text()); continue
        t0 = time.time()
        try:
            j = elevenlabs(path)
        except Exception as e:
            print(f"  ✗ {nombre}: {e}"); continue
        words = norm_elevenlabs(j)
        res[nombre] = words
        crudo.write_text(json.dumps(words, ensure_ascii=False))
        (SAL / f"{nombre}.txt").write_text(
            por_turnos(marcar_censura(words, censura[path.name])), encoding="utf-8")
        print(f"  ✓ {nombre}: {len(words)} palabras en {time.time()-t0:.0f}s")
    return res

# si RES no existe todavía (p.ej. no se corrió 9c6f553a en este kernel), lo reconstruye
# desde los .json que ya están en salidas/ -- deepgram no se vuelve a pagar
if "RES" not in dir():
    RES = {SAL_json.stem: json.loads(SAL_json.read_text()) for SAL_json in SAL.glob("*.json") if SAL_json.stem != "censura"}

RES.update(correr_elevenlabs())

  ✓ ia_corto__elevenlabs: 75 palabras en 3s
  ✓ ia_mediano__elevenlabs: 280 palabras en 5s
  ✓ ia_largo__elevenlabs: 1457 palabras en 10s
  ✓ humano_corto__elevenlabs: 204 palabras en 3s
  ✓ humano_mediano__elevenlabs: 315 palabras en 5s
  ✓ humano_largo__elevenlabs: 3211 palabras en 17s


Cada corrida deja dos archivos en `salidas/`: el `.json` normalizado y un `.txt` legible
ya con los `[CENSURADO]` cruzados y agrupado por turnos de hablante. **Ese `.txt` es el que
tienes que leer** para juzgar cuál motor transcribe mejor.

### 4.4 Comparación cuantitativa

In [16]:
from difflib import SequenceMatcher

def texto(ws): return " ".join(w["text"] for w in ws).lower()
def parecido(a, b): return round(SequenceMatcher(None, texto(a), texto(b)).ratio() * 100, 1)

if RES:
    filas = []
    for nombre, ws in RES.items():
        caso, motor = nombre.split("__")
        confs = [w["conf"] for w in ws if w["conf"] is not None]
        filas.append({"caso": caso, "motor": motor,
                      "palabras": len(ws), "hablantes": len({w["speaker"] for w in ws}),
                      "conf_media": round(np.mean(confs), 3) if confs else None,
                      "pct_conf_baja": round(np.mean([c < .6 for c in confs]) * 100, 1) if confs else None})
    tabla = pd.DataFrame(filas).pivot_table(index="caso", columns="motor").round(3)
    display(tabla)

    print("\nSimilitud de texto entre motores (%):")
    for caso in rutas:
        d, e = RES.get(f"{caso}__deepgram"), RES.get(f"{caso}__elevenlabs")
        if d and e:
            print(f"  {caso:15s} {parecido(d, e):5.1f}")

conf_media            hablantes            palabras             \
motor            deepgram elevenlabs  deepgram elevenlabs deepgram elevenlabs   
caso                                                                            
humano_corto        0.945      0.978       1.0        2.0    181.0      204.0   
humano_largo        0.885      0.963       3.0        2.0   2869.0     3211.0   
humano_mediano      0.899      0.966       3.0        2.0    310.0      315.0   
ia_corto            0.978      0.973       1.0        2.0     67.0       75.0   
ia_largo            0.942      0.980       2.0        2.0   1369.0     1457.0   
ia_mediano          0.974      0.985       1.0        2.0    237.0      280.0   

               pct_conf_baja             
motor               deepgram elevenlabs  
caso                                     
humano_corto             4.4        2.0  
humano_largo            10.4        3.2  
humano_mediano           8.4        2.2  
ia_corto                 1.5        2.7  
ia_largo                 4.0        1.6  
ia_mediano               1.3        1.1


Similitud de texto entre motores (%):
  ia_corto         13.9
  ia_mediano       74.5
  ia_largo         43.1
  humano_corto     49.2
  humano_mediano   19.2
  humano_largo     24.2


Deep: mete a terceras personas muy nada que ver, lo bueno es que tiene numeros

11: la forma en la que maneja los turnos es muy buena en comparacion a deep, como que deep monta conversaciones en turnos que no van